# H&N vs Lung – Phase 1: data facts and linkage coverage (Rybrevant only)

**Purpose:** find out, from the data alone, what can be linked and how well. Nothing in this notebook classifies patients as Lung or H&N.
That needs the diagnosis code lists from Siddhant; section 6 produces the code inventory he needs.

**Scope:** Rybrevant products only (Rybrevant IV + Rybrevant Faspro, every version found in the data). **Lazcluze is excluded everywhere.**

**Filters already applied upstream** (`RYB_HN_Data_Sources.csv`), so they are **not** repeated here:

| Table | Already filtered on |
|---|---|
| DDD | product name contains RYBREVANT or LAZCLUZE |
| Rx claims | Rybrevant / Lazcluze NDCs, `claim_type = PAID`, `claim_status = S` |
| Mx claims | Rybrevant / Lazcluze NDCs, `service_date >= 2025-01-01` |
| OPAL | relation type contains ONC |
| dmn_cust | `end_dt = 2099-12-31`, `cust_stat_nm = Active` |

**Links checked:**
```
Claims NPI ──► dmn_cust (npi_id → jnj_id) ──► OPAL column `0` (HCP) ──► L2_L3 account
DDD jnj_id ──► OPAL column `1` (outlet) / `2` / `3` ─────────────────► L2_L3 account
Claims billing / pharmacy NPI ──► dmn_cust ──► jnj_id ──► is it a DDD buyer?
```
Sections: 1 load · 2 profile · 3 keep Rybrevant · 4 doctor link · 5 sales link · 6 organization link · 7 diagnosis code inventory · 8 account table · 9 ChargeMaster · 10 open questions

## 1. Load data

### What this cell does: load the data
- Reads the six Dataiku datasets into Spark DataFrames:
  - `mx`: medical claims
  - `rx`: pharmacy claims
  - `ddd`: DDD sales
  - `opal`: OPAL hierarchy
  - `cm`: ChargeMaster
  - `dmn`: dmn_cust, the NPI → J&J ID bridge
- `dmn_cust` is in project TA_HCP_UNIVERSER_VERIFY. If it is not shared into H&N_Project, write it as `"PROJECTKEY.dmn_cust"` (use that project's key).
- Prints the row and column count of each table.

In [ ]:
import dataiku
from dataiku import spark as dkuspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
spark = SparkSession.builder.getOrCreate()

def load(name):
    return dkuspark.get_dataframe(spark, dataiku.Dataset(name))

mx   = load("MX_RYB_CLAIMS_copy")
rx   = load("RYB_LINK_CLAIMS_copy")
ddd  = load("RYB_DDD_Sales_copy")
opal = load("OPAL_Affiliation_Levels")
cm   = load("ChargeMaster_WeeklyReporting_20261012_v2_copy")
dmn  = load("dmn_cust")

for name, df in [("mx", mx), ("rx", rx), ("ddd", ddd), ("opal", opal), ("chargemaster", cm), ("dmn_cust", dmn)]:
    print(f"{name:13s} rows={df.count():>12,}  cols={len(df.columns)}")

## 2. Profile each table

### What this cell does: basic facts per table (nothing is filtered)
- For each table it shows:
  - the number of rows;
  - the date range (earliest and latest date);
  - the number of distinct values of each ID column;
  - how many rows have that ID empty.
- Patients in the claims are counted as distinct `dv1` + `dv2`.
- **Look at:** the date ranges (what period we really have), and the empty-ID counts (they limit how much can be linked).

In [ ]:
def blank(c):
    col = F.trim(F.col(f"`{c}`").cast("string"))
    return col.isNull() | (col == "")

def profile(name, df, date_cols, id_cols):
    aggs = [F.count("*").alias("rows")]
    for c in date_cols:
        aggs += [F.min(F.to_date(F.col(f"`{c}`"))).cast("string").alias(f"min {c}"),
                 F.max(F.to_date(F.col(f"`{c}`"))).cast("string").alias(f"max {c}")]
    for c in id_cols:
        aggs += [F.countDistinct(F.col(f"`{c}`")).alias(f"distinct {c}"),
                 F.sum(blank(c).cast("int")).alias(f"empty {c}")]
    print(f"\n=== {name} ===")
    df.agg(*aggs).show(vertical=True, truncate=False)

profile("MX claims", mx.withColumn("patient", F.concat_ws("|", "dv1", "dv2")), ["service_date"],
        ["patient", "mx_claim_id", "ndc", "procedure_code", "rendering_npi", "referring_npi", "billing_npi", "facility_npi"])
profile("RX claims", rx.withColumn("patient", F.concat_ws("|", "dv1", "dv2")), ["fill_date", "written_date"],
        ["patient", "claim_id", "ndc", "prescriber_npi", "pharmacy_npi"])
profile("DDD", ddd, ["caln_dt"], ["jnj_id", "prod_lst_id", "prod_nm"])
profile("OPAL", opal, [], ["0", "1", "2", "3", "4", "5", "L2_L3"])
profile("dmn_cust", dmn, ["efftv_dt", "end_dt"], ["npi_id", "jnj_id"])

## 3. Keep Rybrevant only (drop Lazcluze)

### What this cell does: identify the product on every row and drop Lazcluze
- **Claims:** the product comes from the **NDC reference in `RYB_HN_Data_Sources.csv`**.
  - Rybrevant NDCs: 57894050100, 57894050101.
  - Rybrevant Faspro NDCs: 57894051001, 57894051401, 57894051501, 57894052201.
  - If a claim's NDC is not in that list, the claim's own `product_name` is used when it says RYBREVANT.
  - Anything else (Lazcluze NDCs, other) is dropped.
- **DDD:** keeps rows whose `prod_nm` contains RYBREVANT (Rybrevant and Rybrevant Faspro); Lazcluze rows are dropped.
- Shows what was kept and what was dropped, so you can see every NDC / product name that exists.
- **Look at:** the "dropped" table. It should contain only Lazcluze. Any Rybrevant-looking row there means an NDC missing from the reference.

In [ ]:
RYB_NDC = {   # from RYB_HN_Data_Sources.csv (NDC Reference)
    "57894050100": "RYBREVANT", "57894050101": "RYBREVANT",
    "57894051001": "RYBREVANT FASPRO", "57894051401": "RYBREVANT FASPRO",
    "57894051501": "RYBREVANT FASPRO", "57894052201": "RYBREVANT FASPRO",
}
ndc_map = F.create_map(*[F.lit(x) for kv in RYB_NDC.items() for x in kv])

def ryb_product(ndc_col, name_col):
    name = F.upper(F.trim(F.col(name_col)))
    from_name = F.when(name.contains("RYBREVANT"), name)        # the claim's own product name
    return F.coalesce(ndc_map[F.trim(F.col(ndc_col))], from_name)

mx_p = mx.withColumn("product", ryb_product("ndc", "product_name"))
rx_p = rx.withColumn("product", ryb_product("ndc", "product_name"))

print("MX claims: kept vs dropped by NDC / procedure code / product name")
(mx_p.groupBy(F.when(F.col("product").isNotNull(), "kept").otherwise("dropped").alias("status"),
              "ndc", "procedure_code", "product_name", "product").count().orderBy("status", F.desc("count")).show(50, False))
print("RX claims: kept vs dropped by NDC / product name")
(rx_p.groupBy(F.when(F.col("product").isNotNull(), "kept").otherwise("dropped").alias("status"),
              "ndc", "product_name", "product").count().orderBy("status", F.desc("count")).show(50, False))

ddd_p = ddd.withColumn("product", F.upper(F.trim("prod_nm")))
print("DDD: kept vs dropped by product")
(ddd_p.groupBy(F.when(F.col("product").contains("RYBREVANT"), "kept").otherwise("dropped").alias("status"),
               "prod_lst_id", "prod_nm")
      .agg(F.count("*").alias("rows"), F.round(F.sum("equiv_uns"), 2).alias("equiv_uns"))
      .orderBy("status", "prod_nm").show(50, False))

mx_r  = mx_p.filter(F.col("product").isNotNull())
rx_r  = rx_p.filter(F.col("product").isNotNull())
ddd_r = ddd_p.filter(F.col("product").contains("RYBREVANT"))
print(f"kept rows -> MX {mx_r.count():,} of {mx.count():,} | RX {rx_r.count():,} of {rx.count():,} | DDD {ddd_r.count():,} of {ddd.count():,}")

### What this cell does: remove pharmacy claims of patients who are already in the medical claims
- **Rule (Ravi):** a patient found in the medical claims is taken from the medical claims only. Their pharmacy claims are dropped, so no patient is counted twice.
- Patients who appear **only** in the pharmacy claims are kept from the pharmacy claims (doctor = `prescriber_npi`).
- A patient is `dv1` + `dv2`, the same tokens in both tables.
- Prints:
  - the RX patients and rows before the rule;
  - how many of them are also in MX (dropped);
  - how many are RX-only (kept).
- From here on, `rx_r` holds only the RX-only patients.

In [ ]:
mx_patients = mx_r.select(F.concat_ws("|", "dv1", "dv2").alias("patient_id")).distinct()
rx_all = rx_r.withColumn("patient_id", F.concat_ws("|", "dv1", "dv2"))
rx_in_mx = rx_all.join(mx_patients, "patient_id", "left_semi")
rx_r = rx_all.join(mx_patients, "patient_id", "left_anti").drop("patient_id").cache()

print(f"RX before rule : {rx_all.select('patient_id').distinct().count():,} patients, {rx_all.count():,} rows")
print(f"  also in MX   : {rx_in_mx.select('patient_id').distinct().count():,} patients, {rx_in_mx.count():,} rows  -> dropped")
print(f"  RX only      : {rx_r.select(F.concat_ws('|', 'dv1', 'dv2')).distinct().count():,} patients, {rx_r.count():,} rows  -> kept")

### What this cell does: put every NPI on the claims into one long list
- A claim carries several NPIs. **The doctor on a medical claim is `rendering_npi` (confirmed by Ravi).** All NPIs are still kept, each with its role, so the other roles stay visible for checks:
  - medical claims: `rendering_npi`, `referring_npi`, `billing_npi`, `facility_npi`;
  - pharmacy claims: `prescriber_npi`, `pharmacy_npi`.
- Result `claim_npis`: one row per claim × NPI role, with patient, date and product.
- Later sections report coverage **separately for each role**, so the team can choose with the numbers in front of them.

In [ ]:
def npi_rows(df, source, claim_id, date_col, roles):
    parts = []
    for role in roles:
        parts.append(df.select(F.lit(source).alias("source"), F.col(claim_id).alias("claim_id"),
                               F.concat_ws("|", "dv1", "dv2").alias("patient_id"),
                               F.to_date(date_col).alias("service_dt"), "product",
                               F.lit(role).alias("npi_role"), F.trim(F.col(role).cast("string")).alias("npi")))
    out = parts[0]
    for p in parts[1:]:
        out = out.unionByName(p)
    return out.filter(F.col("npi").isNotNull() & (F.col("npi") != ""))

claim_npis = (npi_rows(mx_r, "MX", "mx_claim_id", "service_date",
                       ["rendering_npi", "referring_npi", "billing_npi", "facility_npi"])
              .unionByName(npi_rows(rx_r, "RX", "claim_id", "fill_date", ["prescriber_npi", "pharmacy_npi"]))
              .cache())
(claim_npis.groupBy("source", "npi_role")
           .agg(F.count("*").alias("claim_rows"), F.countDistinct("npi").alias("distinct_npis"),
                F.countDistinct("patient_id").alias("patients"))
           .orderBy("source", "npi_role").show(truncate=False))

## 4. Doctor link: claims NPI → dmn_cust → OPAL → L2_L3 account

### What this cell does: build the NPI → J&J ID lookup from dmn_cust
- Takes `npi_id` → `jnj_id` from dmn_cust, with customer type and primary specialty for information.
- **Business rule from the data-sources file:** if one NPI has several `jnj_id`s, keep the one with the **latest `efftv_dt`**.
- Prints how many NPIs had more than one J&J ID.
- **Check only:** `cust_stat_nm` and `end_dt` value counts, which should be only Active / 2099-12-31 because of the upstream filter.

In [ ]:
dmn.groupBy("cust_stat_nm").count().show(truncate=False)
dmn.groupBy(F.to_date("end_dt").alias("end_dt")).count().orderBy(F.desc("count")).show(5)

d = (dmn.select(F.trim(F.col("npi_id").cast("string")).alias("npi"), F.col("jnj_id").cast("string").alias("jnj_id"),
                F.to_date("efftv_dt").alias("efftv_dt"), "cust_super_typ_nm", "cust_typ_nm", "prim_spclt_nm")
        .filter("npi IS NOT NULL AND npi != '' AND jnj_id IS NOT NULL"))
npi_jnj = (d.withColumn("n_jnj_ids", F.size(F.collect_set("jnj_id").over(Window.partitionBy("npi"))))
            .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(F.desc_nulls_last("efftv_dt"), "jnj_id")))
            .filter("rk = 1").drop("rk").cache())
print("NPIs in dmn_cust:", npi_jnj.count(), "| NPIs with more than one jnj_id:", npi_jnj.filter("n_jnj_ids > 1").count())

### What this cell does: prepare OPAL
- OPAL columns are read as text:
  - `0` = HCP J&J ID;
  - `1`..`5` = parent ID at each level;
  - `L2_L3` = the account (L2 if present, else L3, as delivered).
- **OPAL facts:**
  - rows and distinct HCPs;
  - HCPs on more than one row;
  - rows with L2 vs only L3 vs no account;
  - whether `L2_L3` really equals "L2 if present, else L3".
- `hcp_accounts`: each HCP with its distinct `L2_L3` accounts, and how many accounts each HCP has.

In [ ]:
def txt(c):
    col = F.trim(F.col(f"`{c}`").cast("string"))
    return F.when(col != "", col).alias(c)

o = opal.select(*[txt(c) for c in ["0", "1", "2", "3", "4", "5", "L2_L3"]], F.col("child_lvl").cast("int").alias("child_lvl"))
o.groupBy("child_lvl").count().show()
o.agg(F.count("*").alias("rows"), F.countDistinct("0").alias("distinct_hcps"),
      F.sum(F.col("2").isNotNull().cast("int")).alias("rows_with_L2"),
      F.sum((F.col("2").isNull() & F.col("3").isNotNull()).cast("int")).alias("rows_L3_only"),
      F.sum(F.col("L2_L3").isNull().cast("int")).alias("rows_without_L2_L3"),
      F.sum((F.coalesce("2", "3") != F.col("L2_L3")).cast("int")).alias("L2_L3_not_equal_L2_else_L3")).show()

hcp_accounts = (o.filter(F.col("0").isNotNull() & F.col("L2_L3").isNotNull())
                 .select(F.col("0").alias("jnj_id"), F.col("L2_L3").alias("account_id")).distinct()
                 .withColumn("n_accounts", F.count("*").over(Window.partitionBy("jnj_id"))).cache())
hcp_accounts.select("jnj_id", "n_accounts").distinct().groupBy("n_accounts").count().orderBy("n_accounts").show()

### What this cell does: doctor-link coverage, per NPI role
- For each NPI role on the claims, counts how many distinct NPIs:
  1. exist on the claims;
  2. are found in **dmn_cust** (get a J&J ID);
  3. have that J&J ID in **OPAL** column `0`;
  4. reach at least one **L2_L3 account**.
- Also shows the dmn_cust customer type of the matched NPIs (doctor vs organization).
- **Look at:** the % reaching an account for **`rendering_npi`**, the doctor on medical claims (confirmed by Ravi), and `prescriber_npi` (pharmacy, RX-only patients). This is the key number for the doctor-to-account link.
  - The other roles are kept for information only.
  - Medical claims with an empty `rendering_npi` have no doctor; section 2 shows how many there are (`empty rendering_npi`).

In [ ]:
opal_hcps = o.filter(F.col("0").isNotNull()).select(F.col("0").alias("jnj_id")).distinct().withColumn("in_opal", F.lit(1))
acct_hcps = hcp_accounts.select("jnj_id").distinct().withColumn("has_account", F.lit(1))

npi_cov = (claim_npis.select("source", "npi_role", "npi").distinct()
           .join(npi_jnj.select("npi", "jnj_id", "cust_typ_nm"), "npi", "left")
           .join(opal_hcps, "jnj_id", "left").join(acct_hcps, "jnj_id", "left"))
(npi_cov.groupBy("source", "npi_role")
        .agg(F.count("*").alias("npis"),
             F.sum(F.col("jnj_id").isNotNull().cast("int")).alias("in_dmn_cust"),
             F.sum(F.coalesce("in_opal", F.lit(0))).alias("in_opal"),
             F.sum(F.coalesce("has_account", F.lit(0))).alias("reach_L2_L3"))
        .withColumn("pct_reach_L2_L3", F.round(100 * F.col("reach_L2_L3") / F.col("npis"), 1))
        .orderBy("source", "npi_role").show(truncate=False))
(npi_cov.groupBy("npi_role", F.coalesce("cust_typ_nm", F.lit("NOT IN DMN_CUST")).alias("dmn_cust_type"))
        .count().orderBy("npi_role", F.desc("count")).show(40, False))

## 5. Sales link: DDD jnj_id → OPAL → L2_L3 account

### What this cell does: find where the DDD buyers sit in OPAL
- Lists every ID in OPAL with the column(s) it appears in (`0` to `5`).
- Looks up each DDD `jnj_id` (Rybrevant rows only) and groups the DDD units by what was found:
  - **L1 → 1 account**: the ID is an outlet in OPAL column `1` and all its rows point to one `L2_L3`;
  - **L1 → several accounts**: the outlet points to more than one `L2_L3` (needs a rule, see section 10);
  - **is L2 / is L3**: the ID is itself an account in column `2` or `3`;
  - **other OPAL level** / **not in OPAL**.
- **Look at:** the % of units that can be placed on exactly one account.

In [ ]:
opal_ids = None
for lvl in ["0", "1", "2", "3", "4", "5"]:
    part = o.filter(F.col(lvl).isNotNull()).select(F.col(lvl).alias("jnj_id"), F.lit(lvl).alias("col"))
    opal_ids = part if opal_ids is None else opal_ids.unionByName(part)
opal_ids = (opal_ids.distinct().groupBy("jnj_id")
            .agg(F.concat_ws("+", F.sort_array(F.collect_set("col"))).alias("opal_columns")))

outlet_acct = (o.filter(F.col("1").isNotNull() & F.col("L2_L3").isNotNull())
                .groupBy(F.col("1").alias("jnj_id"))
                .agg(F.countDistinct("L2_L3").alias("n_accounts"), F.min("L2_L3").alias("only_account")))

ddd_ids = (ddd_r.groupBy(F.col("jnj_id").cast("string").alias("jnj_id"))
                .agg(F.sum("equiv_uns").alias("equiv_uns"))
                .join(opal_ids, "jnj_id", "left").join(outlet_acct, "jnj_id", "left"))
ddd_ids = ddd_ids.withColumn("link",
    F.when(F.col("n_accounts") == 1, "L1 -> 1 account")
     .when(F.col("n_accounts") > 1, "L1 -> several accounts")
     .when(F.col("opal_columns").contains("2"), "is L2")
     .when(F.col("opal_columns").contains("3"), "is L3")
     .when(F.col("opal_columns").isNotNull(), "other OPAL column: " )
     .otherwise("not in OPAL"))
ddd_ids = ddd_ids.withColumn("link", F.when(F.col("link") == "other OPAL column: ",
                                            F.concat(F.col("link"), F.col("opal_columns"))).otherwise(F.col("link"))).cache()
(ddd_ids.groupBy("link").agg(F.count("*").alias("ddd_ids"), F.round(F.sum("equiv_uns"), 2).alias("equiv_uns"))
        .withColumn("pct_units", F.round(100 * F.col("equiv_uns") / F.sum("equiv_uns").over(Window.partitionBy()), 1))
        .orderBy(F.desc("equiv_uns")).show(truncate=False))

## 6. Organization link: billing / pharmacy NPI → dmn_cust → DDD

### What this cell does: check whether claim organizations are DDD buyers
- Takes the organization NPIs on the claims: `billing_npi`, `facility_npi` (medical) and `pharmacy_npi` (pharmacy).
- Looks them up in dmn_cust to get a J&J ID, then checks:
  - is that J&J ID a Rybrevant **DDD buyer**?
  - in which **OPAL** column does it appear?
- **Look at:** if many billing NPIs are DDD buyers, claims can be linked to sales directly. That would be a second route, a cross-check of the doctor route.

In [ ]:
ddd_buyers = ddd_r.select(F.col("jnj_id").cast("string").alias("jnj_id")).distinct().withColumn("is_ddd_buyer", F.lit(True))
org = (claim_npis.filter(F.col("npi_role").isin("billing_npi", "facility_npi", "pharmacy_npi"))
       .groupBy("npi_role", "npi").agg(F.countDistinct("patient_id").alias("patients"))
       .join(npi_jnj.select("npi", "jnj_id", "cust_typ_nm"), "npi", "left")
       .join(ddd_buyers, "jnj_id", "left").join(opal_ids, "jnj_id", "left")
       .fillna({"is_ddd_buyer": False}))
(org.groupBy("npi_role", F.col("jnj_id").isNotNull().alias("in_dmn_cust"), "is_ddd_buyer", "opal_columns")
    .agg(F.count("*").alias("org_npis"), F.sum("patients").alias("patients"))
    .orderBy("npi_role", F.desc("org_npis")).show(40, False))

## 7. Diagnosis code inventory (for Siddhant)

### What this cell does: list every diagnosis code on the Rybrevant claims, with no classification
- Medical claims: `diagnosis_codes` is a `|`-separated list, so it is split into single codes.
- Pharmacy claims: `diagnosis_code`, for RX-only patients.
- For each code: number of claims and patients, by source (MX / RX) and product.
- Also: how many claims and patients have **no** diagnosis code.
- **Nothing is labelled Lung or H&N here.** This table goes to Siddhant, who marks which codes are Lung, H&N, or neither. His list then drives the labelling.
- To export it, see the save cell at the end (`dx_inventory`).

In [ ]:
mx_dx = mx_r.select(F.lit("MX").alias("source"), F.col("mx_claim_id").alias("claim_id"),
                    F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "product",
                    F.explode_outer(F.split(F.coalesce(F.col("diagnosis_codes"), F.lit("")), r"\|")).alias("dx_code"))
rx_dx = rx_r.select(F.lit("RX").alias("source"), "claim_id",
                    F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "product",
                    F.col("diagnosis_code").alias("dx_code"))
dx = mx_dx.unionByName(rx_dx).withColumn("dx_code", F.upper(F.trim("dx_code")))

no_dx = (dx.groupBy("source", "claim_id", "patient_id")
           .agg(F.max(F.when(F.col("dx_code") != "", 1).otherwise(0)).alias("has_dx")))
(no_dx.groupBy("source").agg(F.count("*").alias("claims"),
                             F.sum((F.col("has_dx") == 0).cast("int")).alias("claims_without_dx")).show())

dx_inventory = (dx.filter(F.col("dx_code").isNotNull() & (F.col("dx_code") != ""))
                  .groupBy("dx_code", F.substring("dx_code", 1, 3).alias("dx_3char"), "source", "product")
                  .agg(F.countDistinct("claim_id").alias("claims"), F.countDistinct("patient_id").alias("patients"))
                  .orderBy(F.desc("patients")).cache())
print("distinct diagnosis codes:", dx_inventory.select("dx_code").distinct().count())
dx_inventory.show(100, False)

## 8. Account table (no Lung / H&N labels yet)

### What this cell does: put DDD units and claim doctors side by side per L2_L3 account
- **Sales side:** Rybrevant and Rybrevant Faspro DDD units per account.
  - Only DDD IDs with **one** clear account are used: "L1 → 1 account", "is L2", "is L3".
  - Ambiguous IDs are left out and counted in section 5.
- **Doctor side:** for each account, the number of distinct doctors that reach it, and their patients:
  - medical claims: **`rendering_npi`** (confirmed by Ravi);
  - pharmacy claims: `prescriber_npi`, only for RX-only patients (patients already in MX were removed in section 3), shown in its own column.
  - A doctor with several accounts is counted **fully at each one** (no split). How to split them is an open question.
- Then: how many accounts have sales and doctors, sales only, or doctors only, with their share of units.
- **Look at:** the share of units in accounts that also have claim doctors. That is how much of the sales can later be split by the doctors' Lung / H&N mix.

In [ ]:
ddd_acct_map = (ddd_ids.filter(F.col("link").isin("L1 -> 1 account", "is L2", "is L3"))
                       .select("jnj_id", F.when(F.col("link") == "L1 -> 1 account", F.col("only_account"))
                                          .otherwise(F.col("jnj_id")).alias("account_id")))
acct_sales = (ddd_r.withColumn("jnj_id", F.col("jnj_id").cast("string")).join(ddd_acct_map, "jnj_id")
              .groupBy("account_id")
              .pivot("product").agg(F.round(F.sum("equiv_uns"), 3)).fillna(0))
unit_cols = [c for c in acct_sales.columns if c != "account_id"]
acct_sales = acct_sales.withColumn("total_ryb_units", sum(F.col(f"`{c}`") for c in unit_cols))

hcp_roles = ["rendering_npi", "prescriber_npi"]   # doctor = rendering NPI on medical claims (Ravi); prescriber on pharmacy claims
acct_docs = (claim_npis.filter(F.col("npi_role").isin(hcp_roles))
             .join(npi_jnj.select("npi", "jnj_id"), "npi").join(hcp_accounts, "jnj_id")
             .groupBy("account_id").pivot("npi_role", hcp_roles)
             .agg(F.countDistinct("npi").alias("npis"), F.countDistinct("patient_id").alias("patients"))
             .fillna(0))

accounts = acct_sales.join(acct_docs, "account_id", "full").fillna(0)
doc_cols = [c for c in accounts.columns if c.endswith("_npis")]
accounts = accounts.withColumn("has_sales", F.col("total_ryb_units") > 0) \
                   .withColumn("has_doctors", sum(F.col(c) for c in doc_cols) > 0)
(accounts.groupBy("has_sales", "has_doctors")
         .agg(F.count("*").alias("accounts"), F.round(F.sum("total_ryb_units"), 2).alias("units"))
         .withColumn("pct_units", F.round(100 * F.col("units") / F.sum("units").over(Window.partitionBy()), 1))
         .orderBy(F.desc("units")).show())
accounts.orderBy(F.desc("total_ryb_units")).show(25, False)

## 9. ChargeMaster, as reported

### What this cell does: read the ChargeMaster report without interpreting it
- All columns are stored as text, so the numbers are converted to numbers.
- Shows which cadences exist (annual, monthly, …), then the annual rows: patients, encounters (all four blocks: no suffix, `_1`, `_2`, `_3`), and the two share sets (`NSCLC/HNSCC/Other` and `NSCLC_1/HNSCC_1/Other_1`).
- **Consistency facts:** whether encounters = inpatient + outpatient in each block, and what each share set adds up to.
- The meaning of the `_1/_2/_3` blocks and of the two share sets is **not known yet**. It is an open question for Ryan.

In [ ]:
num = lambda c: F.regexp_replace(F.col(f"`{c}`").cast("string"), ",", "").cast("double")
cm_c = cm.select(F.col("`Time Frame (Sorter)`").alias("tf"), F.col("`Time Frame Descriptive`").alias("tf_desc"), "Cadence",
                 *[num(f"# {m}{s}").alias(f"{m.lower()}{s}") for s in ["", "_1", "_2", "_3"]
                   for m in ["Patients", "Encounters", "EncountersInpatient", "EncountersOutpatient"]],
                 *[num(c).alias(c.lower()) for c in ["NSCLC", "HNSCC", "Other", "NSCLC_1", "HNSCC_1", "Other_1"]])
cm_c.groupBy("Cadence").count().show(truncate=False)

checks = cm_c.select("tf", "tf_desc", "Cadence",
    *[(F.col(f"encounters{s}") - F.col(f"encountersinpatient{s}") - F.col(f"encountersoutpatient{s}")).alias(f"enc_gap{s or '_base'}")
      for s in ["", "_1", "_2", "_3"]],
    (F.col("nsclc") + F.col("hnscc") + F.col("other")).alias("share_sum"),
    (F.col("nsclc_1") + F.col("hnscc_1") + F.col("other_1")).alias("share_sum_1"))
checks.orderBy("tf").show(50, False)

(cm_c.filter(F.upper("Cadence").contains("ANNUAL"))
     .select("tf_desc", "patients", "encounters", "encounters_1", "encounters_2", "encounters_3",
             "nsclc", "hnscc", "other", "nsclc_1", "hnscc_1", "other_1")
     .orderBy("tf_desc").show(truncate=False))

## 10. Open questions

Kept in **`OPEN_QUESTIONS.md`** (next to this notebook), the single place that is updated as answers come in.
That file has three parts: settled decisions, open questions (with the numbers from this notebook to bring), and cases to check in the numbers before asking.

### What this cell does: optionally save the key tables to Dataiku
- Leave `SAVE = False` to only look at results.
- To save: create empty output datasets in the Flow with these names, then set `SAVE = True`.
- `HN_DX_CODE_INVENTORY` is the table to hand to Siddhant.

In [ ]:
SAVE = False
outputs = {"HN_DX_CODE_INVENTORY": dx_inventory, "HN_ACCOUNT_TABLE": accounts,
           "HN_NPI_COVERAGE": npi_cov, "HN_DDD_ID_LINK": ddd_ids}
if SAVE:
    for name, df in outputs.items():
        dkuspark.write_with_schema(dataiku.Dataset(name), df)
        print("saved", name)